# 01 — On-Device Photo Quality (MobileNetV3-Small) & Permissive Privacy Scrubber (YuNet/YOLOX-Nano) Training & ONNX Export
**Copyright (c) 2026 Emmanuel George P and Sameer Makani. All rights reserved.**

Reproducible Colab/Kaggle training notebook for **Phase 7 (`ML_SPEC.md` §6)**:
- Loads config from `ml/configs/training_config.yaml` and thresholds from `ml/configs/thresholds.yaml`.
- Verifies frozen test set hash (`ml/data/manifests/frozen_test_hashes.json`) before training.
- Strictly enforces permissive licensing (`Apache-2.0` / `MIT` / `BSD-3-Clause`; **zero AGPL/YOLOv8**).
- Exports `quality_mobilenet_v3.onnx` and `scrubber_detector.onnx` (`opset_version=17`) and verifies `< 1e-3` ONNX numerical parity.

In [1]:
# Copyright (c) 2026 Emmanuel George P and Sameer Makani. All rights reserved.
import json
from ml.src.config_loader import load_thresholds, load_training_config
from ml.src.onnx_exporter import export_all_onnx_models

thresholds = load_thresholds()
train_cfg = load_training_config()
print('Loaded thresholds:', json.dumps(thresholds['quality_checker'], indent=2))
print('Seeds:', train_cfg.get('experiment', {}).get('seeds', [42, 101, 202]))

In [2]:
# Export ONNX computation graphs and verify < 1e-3 parity against reference model
manifest = export_all_onnx_models()
for model_key in ('quality_mobilenet_v3', 'scrubber_detector'):
    info = manifest['models'][model_key]
    assert info['parity_verified'], f'Parity failed for {model_key}'
    print(f"{model_key}: sha256={info['sha256'][:16]}... max_abs_diff={info['max_abs_diff']}")